[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/D.%20Strategic%20Network%20Design%20%26%20Sourcing%20%28The%20Blueprint%20of%20the%20Business%29/091.%20The%20Contract%20Design%20to%20Mitigate%20the%20Bullwhip%20Effect/P91-Tier-3_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 91. Contract Design to Mitigate the Bullwhip Effect
## Tier 3: Bat-Inspired Population Search for Multi-Party Contract Optimization

### Purpose and scope

This notebook uses a bat-inspired population-based search to choose 25 contract parameters for a five-tier supply chain. Each candidate contract is evaluated with a mathematical model of bullwhip amplification and implementation cost. The search returns a best-found parameter vector, its scalar fitness, and diagnostic measures such as population diversity and convergence history.

The method is a stochastic heuristic: it searches for a good solution but does not provide a proof of global optimality. A fixed random seed makes a run reproducible in a given software environment; it does not make the method deterministic across different seeds or guarantee the same result under different library versions.

### Modeling assumptions

- The supply chain has five tiers, and the decision vector contains exactly 25 continuous parameters.
- The first ten parameters describe information-sharing coefficients; the remaining groups describe cost allocation, penalties, capacity reservation, and timing.
- Bullwhip amplification and implementation cost are represented by the explicit functions in the optimizer. They are stylized model relationships, not estimates fitted to observed supply-chain data.
- The baseline amplification is 4.1. The information-sharing groups are treated asymmetrically in the amplification calculation: the first five coefficients represent downstream sharing and the next five upstream sharing.
- The implementation-cost model uses the coefficients and dollar amounts specified in the code. It does not model cash flows, discounting, uncertainty, or detailed operational constraints.
- The model does not include additional feasibility conditions such as contractual participation, budget allocation between individual partners, or integer decisions.

### Search outline

1. Initialize a population of candidate contracts and their frequencies, loudness values, and pulse rates.
2. Evaluate each candidate using the scalar fitness function.
3. Update each candidate using frequency-modulated moves; use local search around the best-known candidate when the loudness and pulse-rate tests call for it.
4. Accept improvements, update bat parameters and the best-known solution, and record convergence and diversity.
5. Stop at the iteration limit or when the best fitness has changed by less than the specified tolerance over the recent comparison window.

### Interpreting the example

The settings below use five tiers, 40 bats, 25 parameters, and a maximum of 500 iterations. These are experiment settings, not a claim that this population size or iteration count is universally optimal. The notebook reports results produced by the run; it does not assume a particular convergence iteration, information-sharing level, cost, or reduction in amplification.

## Mathematical model and fitness

A candidate contract is a vector \(\mathbf{x}\in\mathbb{R}^{25}\). Its entries are divided into five groups: information sharing, cost allocation, penalty structures, capacity terms, and timing parameters. Each group has the bounds specified in the optimizer. The algorithm keeps candidates inside these bounds by clipping proposed positions.

For a candidate \(\mathbf{x}\), let \(A(\mathbf{x})\) denote modeled bullwhip amplification and \(C(\mathbf{x})\) denote modeled implementation cost in dollars. The code minimizes the following scalar fitness:

$$\
F(\mathbf{x}) = A(\mathbf{x}) + 10\max\{0, A(\mathbf{x})-2.0\}
+ \frac{\max\{0, C(\mathbf{x})-2{,}300{,}000\}}{100{,}000}.
$$

The first term rewards lower amplification. The second adds a penalty when amplification exceeds the target of 2.0. The final term adds a normalized penalty when modeled implementation cost exceeds $2.3 million. Thus, cost is a penalty component rather than a separate objective with its own weight or a Pareto trade-off.

Under the parameter bounds and cost formula in this notebook, the maximum possible modeled cost is approximately $2.212 million: the upper-bound contributions are $800,000 for information systems, $600,000 for cost allocation, about $217,638 for penalty systems, $450,000 for capacity reservation, and $144,000 for timing systems. Consequently, the cost-excess penalty is inactive throughout this bounded search. Cost is still calculated and reported, but this particular formulation does not make the optimizer trade lower cost against lower amplification.

The modeled amplification starts from the baseline \(A_0=4.1\). Define the mean downstream and upstream information-sharing coefficients as \(s_d\) and \(s_u\), respectively. The code computes information effectiveness as \((0.7s_d+0.3s_u)^{0.8}\). It combines that value with the mean-based effectiveness terms for the other four parameter groups, using weights 0.4, 0.2, 0.15, 0.15, and 0.1. If the resulting weighted effectiveness is \(E(\mathbf{x})\), then

$$\
A(\mathbf{x})=\max\left\{4.1\left(1-0.7E(\mathbf{x})\right),\ 1.0\right\}.
$$

The 0.7 multiplier limits the modeled reduction factor, and the maximum enforces a minimum amplification of 1.0. The powers and weights in this relationship are stipulated modeling choices in the code; they are not derived from the bat search itself.

In the implementation, `params` represents \(\mathbf{x}\), `_calculate_bullwhip_amplification` computes \(A(\mathbf{x})\), `_calculate_implementation_cost` computes \(C(\mathbf{x})\), and `_evaluate_fitness` computes \(F(\mathbf{x})\).

## Bat-inspired search: intuition and update logic

The population is a collection of candidate contracts. A bat's position is one 25-parameter vector; its frequency controls the scale and style of the proposed move. Lower frequency values use a broader perturbation phase, while higher values use a move toward the best-known position with a smaller random perturbation. This is the role of frequency modulation in this implementation: it varies the search behavior as candidate positions change.

The local-search step samples five nearby candidates around the best-known position. Loudness and pulse rate control whether that step is attempted. A proposed position replaces a bat's current position only when its fitness is lower. Accepted moves increase that bat's pulse rate up to 1 and reduce its loudness by the specified decay factor. The best-known solution is updated when an accepted candidate improves on the best fitness seen so far.

More formally, let \(\mathbf{x}_i^{(t)}\) be bat \(i\)'s position at iteration \(t\), \(f_i^{(t)}\) its frequency, and \(\mathbf{x}_*^{(t)}\) the best-known position. The implementation samples frequency uniformly from the configured interval. For a frequency factor \(q_i=f_i/f_{\max}\), it proposes a coordinate-wise perturbation when \(q_i<0.5\); otherwise it moves selected coordinates partly toward \(\mathbf{x}_*\) and adds random noise. The proposal is clipped to the parameter bounds. This is a bat-inspired variant with explicit update rules in the code; it should not be interpreted as a claim that every canonical Bat Algorithm implementation uses these exact rules.

The algorithm records the best fitness once per iteration. It also records population diversity, defined as the average Euclidean distance over all distinct pairs of bats:

$$\
D^{(t)}=\frac{2}{B(B-1)}\sum_{1\leq i<j\leq B}
\left\|\mathbf{x}_i^{(t)}-\mathbf{x}_j^{(t)}\right\|_2,
\qquad B>1.
$$

Here, \(B\) is the population size. Greater diversity means the candidate vectors are, on average, farther apart in the raw parameter coordinates; it does not by itself mean that the candidates have better fitness. Because parameter groups have different ranges, this raw Euclidean measure is a diagnostic rather than a scale-normalized distance.

If \(T\) is the number of iterations, \(B\) the number of bats, and \(d=25\) the number of parameters, candidate updates and a fixed number of local-search trials require order \(O(TBd)\) fitness work. The pairwise diversity calculation requires order \(O(TB^2d)\) work. The population, best candidate, and recorded histories are maintained explicitly by the optimizer; no dynamic-programming state or recursion is used.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

# A fixed seed makes the stochastic search reproducible in a consistent environment.
np.random.seed(91)

print("Libraries imported successfully for the bat-inspired search.")

In [ ]:
class BatAlgorithmContractOptimizer:
    """Bat-inspired population search for multi-party contract parameters."""

    def __init__(self, n_tiers=5, n_bats=40, max_iterations=500):
        self.n_tiers = n_tiers
        self.n_bats = n_bats
        self.max_iterations = max_iterations

        # The five parameter groups contain 10, 5, 5, 3, and 2 values.
        self.n_parameters = 25
        self.parameter_bounds = {
            "info_sharing": (0.1, 1.0),
            "cost_allocation": (0.0, 1.0),
            "penalty_structures": (0.05, 0.5),
            "capacity_terms": (0.0, 1.0),
            "timing_parameters": (0.1, 0.9),
        }

        # These settings control frequency-modulated moves and local search.
        self.frequency_min = 0.0
        self.frequency_max = 2.0
        self.initial_loudness = 0.5
        self.initial_pulse_rate = 0.5
        self.loudness_decay = 0.95
        self.pulse_rate_increase = 0.05

        # Cost and demand descriptors retained as model context.
        self.holding_costs = [2.5, 2.0, 1.8, 1.5, 1.2]
        self.stockout_costs = [20.0, 16.0, 14.0, 12.0, 10.0]
        self.procurement_costs = [10.0, 8.0, 7.0, 6.0, 5.0]
        self.base_demand = 2000
        self.demand_variance = 0.4

        self.bats = None
        self.frequencies = None
        self.loudness = None
        self.pulse_rates = None

        self.best_solution = None
        self.best_fitness = float("inf")
        self.convergence_history = []
        self.diversity_history = []

    def _initialize_bat_population(self):
        """Create bounded candidate contracts and evaluate the initial population."""
        self.bats = np.zeros((self.n_bats, self.n_parameters))
        self.frequencies = np.zeros(self.n_bats)
        self.loudness = np.full(self.n_bats, self.initial_loudness)
        self.pulse_rates = np.full(self.n_bats, self.initial_pulse_rate)

        for i in range(self.n_bats):
            self.bats[i] = self._generate_random_contract_params()
            self.frequencies[i] = np.random.uniform(
                self.frequency_min, self.frequency_max
            )

        # The initial best is needed before frequency-based exploitation can use it.
        for i in range(self.n_bats):
            fitness = self._evaluate_fitness(self.bats[i])
            if fitness < self.best_fitness:
                self.best_fitness = fitness
                self.best_solution = self.bats[i].copy()

    def _generate_random_contract_params(self):
        """Sample one 25-parameter contract from the five specified bounds."""
        params = np.zeros(self.n_parameters)
        params[:10] = np.random.uniform(0.1, 1.0, size=10)
        params[10:15] = np.random.uniform(0.0, 1.0, size=5)
        params[15:20] = np.random.uniform(0.05, 0.5, size=5)
        params[20:23] = np.random.uniform(0.0, 1.0, size=3)
        params[23:25] = np.random.uniform(0.1, 0.9, size=2)
        return params

    def _evaluate_fitness(self, params):
        """Return the scalar objective; lower values represent better candidates."""
        if not self._check_parameter_bounds(params):
            return float("inf")

        amplification = self._calculate_bullwhip_amplification(params)
        implementation_cost = self._calculate_implementation_cost(params)

        # Penalties encode the amplification target and the stated cost threshold.
        amplification_penalty = max(0.0, amplification - 2.0) * 10.0
        cost_penalty = max(0.0, implementation_cost - 2_300_000) / 100_000
        return amplification + amplification_penalty + cost_penalty

    def _check_parameter_bounds(self, params):
        """Check that every parameter belongs to its assigned interval."""
        params = np.asarray(params)
        if params.shape != (self.n_parameters,):
            return False

        if np.any(params[:10] < 0.1) or np.any(params[:10] > 1.0):
            return False
        if np.any(params[10:15] < 0.0) or np.any(params[10:15] > 1.0):
            return False
        if np.any(params[15:20] < 0.05) or np.any(params[15:20] > 0.5):
            return False
        if np.any(params[20:23] < 0.0) or np.any(params[20:23] > 1.0):
            return False
        if np.any(params[23:25] < 0.1) or np.any(params[23:25] > 0.9):
            return False
        return True

    def _calculate_bullwhip_amplification(self, params):
        """Evaluate modeled amplification using all five contract parameter groups."""
        info_sharing = params[:10]
        cost_allocation = params[10:15]
        penalty_structures = params[15:20]
        capacity_terms = params[20:23]
        timing_params = params[23:25]

        base_amplification = 4.1
        downstream_sharing = np.mean(info_sharing[:5])
        upstream_sharing = np.mean(info_sharing[5:])

        info_effectiveness = (
            0.7 * downstream_sharing + 0.3 * upstream_sharing
        ) ** 0.8
        cost_effectiveness = np.mean(cost_allocation) ** 0.6
        penalty_effectiveness = np.mean(penalty_structures) ** 0.7
        capacity_benefits = np.mean(capacity_terms) ** 0.5
        timing_benefits = np.mean(timing_params) ** 0.4

        total_effectiveness = (
            0.4 * info_effectiveness
            + 0.2 * cost_effectiveness
            + 0.15 * penalty_effectiveness
            + 0.15 * capacity_benefits
            + 0.1 * timing_benefits
        )

        reduction_factor = 0.7 * total_effectiveness
        final_amplification = base_amplification * (1.0 - reduction_factor)
        return max(final_amplification, 1.0)

    def _calculate_implementation_cost(self, params):
        """Return the modeled implementation cost in dollars."""
        info_costs = np.sum(params[:10] ** 1.5) * 80_000
        allocation_costs = np.sum(params[10:15]) * 120_000
        penalty_costs = np.sum(params[15:20] ** 1.2) * 100_000
        capacity_costs = np.sum(params[20:23]) * 150_000
        timing_costs = np.sum(params[23:25]) * 80_000
        return (
            info_costs
            + allocation_costs
            + penalty_costs
            + capacity_costs
            + timing_costs
        )

    def _update_bat_position(self, bat_idx, iteration):
        """Propose a bounded position using frequency-dependent movement."""
        self.frequencies[bat_idx] = np.random.uniform(
            self.frequency_min, self.frequency_max
        )
        new_position = self.bats[bat_idx].copy()
        frequency_factor = self.frequencies[bat_idx] / self.frequency_max

        if frequency_factor < 0.5:
            # Lower frequencies permit larger random exploration steps.
            step_size = 0.3 * (1.0 - frequency_factor)
            for i in range(self.n_parameters):
                if np.random.random() < 0.3:
                    new_position[i] += np.random.uniform(-step_size, step_size)
        else:
            # Higher frequencies emphasize movement toward the best-known contract.
            step_size = 0.1 * (1.0 - frequency_factor)
            for i in range(self.n_parameters):
                if np.random.random() < 0.5:
                    direction = self.best_solution[i] - self.bats[bat_idx][i]
                    new_position[i] += (
                        direction * step_size + np.random.uniform(-0.05, 0.05)
                    )

        return self._clip_parameters(new_position)

    def _clip_parameters(self, params):
        """Keep every proposed coordinate inside its contract-parameter bounds."""
        params = np.asarray(params).copy()
        params[:10] = np.clip(params[:10], 0.1, 1.0)
        params[10:15] = np.clip(params[10:15], 0.0, 1.0)
        params[15:20] = np.clip(params[15:20], 0.05, 0.5)
        params[20:23] = np.clip(params[20:23], 0.0, 1.0)
        params[23:25] = np.clip(params[23:25], 0.1, 0.9)
        return params

    def _local_search(self, position):
        """Test five small perturbations around a promising candidate."""
        candidates = []
        for _ in range(5):
            candidate = position.copy()
            for i in range(self.n_parameters):
                if np.random.random() < 0.2:
                    candidate[i] += np.random.uniform(-0.02, 0.02)
            candidates.append(self._clip_parameters(candidate))

        best_candidate = position.copy()
        best_fitness = self._evaluate_fitness(position)
        for candidate in candidates:
            fitness = self._evaluate_fitness(candidate)
            if fitness < best_fitness:
                best_fitness = fitness
                best_candidate = candidate.copy()
        return best_candidate

    def _calculate_population_diversity(self):
        """Return mean pairwise Euclidean distance between candidate vectors."""
        if self.n_bats <= 1:
            return 0.0

        total_distance = 0.0
        count = 0
        for i in range(self.n_bats):
            for j in range(i + 1, self.n_bats):
                total_distance += np.linalg.norm(self.bats[i] - self.bats[j])
                count += 1
        return total_distance / count if count else 0.0

    def optimize(self):
        """Run the population search and return the best candidate and fitness."""
        if self.n_bats < 1:
            raise ValueError("The bat population must contain at least one candidate.")
        if self.max_iterations < 1:
            raise ValueError("The iteration limit must be at least one.")

        print("Starting bat-inspired search for contract optimization...")
        print(f"Population size: {self.n_bats} bats")
        print(f"Problem dimension: {self.n_parameters} parameters")
        print(f"Maximum iterations: {self.max_iterations}")
        print()

        self.best_solution = None
        self.best_fitness = float("inf")
        self.convergence_history = []
        self.diversity_history = []
        self._initialize_bat_population()
        start_time = time.time()

        for iteration in range(self.max_iterations):
            for i in range(self.n_bats):
                new_position = self._update_bat_position(i, iteration)
                new_fitness = self._evaluate_fitness(new_position)

                # Loudness and pulse rate determine whether to refine the current best.
                if np.random.random() > self.pulse_rates[i]:
                    if np.random.random() < self.loudness[i]:
                        local_solution = self._local_search(self.best_solution)
                        local_fitness = self._evaluate_fitness(local_solution)
                        if local_fitness < new_fitness:
                            new_position = local_solution
                            new_fitness = local_fitness

                current_fitness = self._evaluate_fitness(self.bats[i])
                if new_fitness < current_fitness:
                    self.bats[i] = new_position.copy()
                    self.pulse_rates[i] = min(
                        1.0, self.pulse_rates[i] + self.pulse_rate_increase
                    )
                    self.loudness[i] *= self.loudness_decay

                    if new_fitness < self.best_fitness:
                        self.best_fitness = new_fitness
                        self.best_solution = new_position.copy()

            self.convergence_history.append(self.best_fitness)
            diversity = self._calculate_population_diversity()
            self.diversity_history.append(diversity)

            if (iteration + 1) % 50 == 0:
                print(
                    f"Iteration {iteration + 1}: "
                    f"Best fitness = {self.best_fitness:.6f}, "
                    f"Diversity = {diversity:.4f}"
                )

            # Compare the latest value with the value 49 recorded steps earlier.
            if iteration > 100 and len(self.convergence_history) > 50:
                recent_improvement = abs(
                    self.convergence_history[-1] - self.convergence_history[-50]
                )
                if recent_improvement < 1e-6:
                    print(f"Converged after {iteration + 1} iterations.")
                    break

        optimization_time = time.time() - start_time
        print(f"\nOptimization completed in {optimization_time:.2f} seconds.")
        print(f"Best fitness: {self.best_fitness:.6f}")
        return self.best_solution.copy(), self.best_fitness

    def analyze_results(self, optimal_params):
        """Report modeled contract outcomes and search diagnostics."""
        if not self._check_parameter_bounds(optimal_params):
            raise ValueError("The supplied contract parameters are outside their bounds.")

        print("\n=== BAT-INSPIRED CONTRACT OPTIMIZATION RESULTS ===\n")
        info_sharing = optimal_params[:10]
        cost_allocation = optimal_params[10:15]
        penalty_structures = optimal_params[15:20]
        capacity_terms = optimal_params[20:23]
        timing_params = optimal_params[23:25]

        final_amplification = self._calculate_bullwhip_amplification(optimal_params)
        implementation_cost = self._calculate_implementation_cost(optimal_params)
        baseline_amplification = 4.1
        reduction_percentage = (
            (baseline_amplification - final_amplification)
            / baseline_amplification
            * 100.0
        )

        print("MULTI-PARTY CONTRACT MODEL:")
        print(f"  Baseline amplification: {baseline_amplification:.2f}x")
        print(f"  Modeled amplification: {final_amplification:.2f}x")
        print(f"  Modeled reduction from baseline: {reduction_percentage:.1f}%")
        print(f"  Modeled implementation cost: ${implementation_cost:,.0f}")

        downstream_sharing = np.mean(info_sharing[:5])
        upstream_sharing = np.mean(info_sharing[5:])
        print("\nASYMMETRIC INFORMATION-SHARING SUMMARY:")
        print(f"  Downstream mean coefficient: {downstream_sharing:.3f}")
        print(f"  Upstream mean coefficient: {upstream_sharing:.3f}")
        print(f"  Downstream-to-upstream ratio: {downstream_sharing / upstream_sharing:.2f}")

        tier_names = ["Tier 1-2", "Tier 2-3", "Tier 3-4", "Tier 4-5", "Cross-tier"]
        print("\nCOST-ALLOCATION PARAMETERS:")
        for name, allocation in zip(tier_names, cost_allocation):
            print(f"  {name}: {allocation:.3f}")

        initial_diversity = (
            self.diversity_history[0] if self.diversity_history else 0.0
        )
        final_diversity = (
            self.diversity_history[-1] if self.diversity_history else 0.0
        )
        diversity_reduction = (
            (1.0 - final_diversity / initial_diversity) * 100.0
            if initial_diversity > 0.0
            else 0.0
        )
        print("\nSEARCH DIAGNOSTICS:")
        print(f"  Recorded iterations: {len(self.convergence_history)}")
        print(f"  Initial population diversity: {initial_diversity:.4f}")
        print(f"  Final population diversity: {final_diversity:.4f}")
        print(f"  Diversity reduction: {diversity_reduction:.1f}%")

        return {
            "amplification": final_amplification,
            "reduction_percentage": reduction_percentage,
            "impl_cost": implementation_cost,
            "downstream_sharing": downstream_sharing,
            "upstream_sharing": upstream_sharing,
            "info_sharing": info_sharing.copy(),
            "cost_allocation": cost_allocation.copy(),
            "penalty_structures": penalty_structures.copy(),
            "capacity_terms": capacity_terms.copy(),
            "timing_params": timing_params.copy(),
        }


print("BatAlgorithmContractOptimizer class defined successfully.")

## Run the main experiment

This cell creates the optimizer with the example settings and runs the population search. Each printed fitness is the scalar objective defined above, not amplification alone. The returned solution is the best candidate found by this run; a heuristic search result is not a certificate of a global optimum.

In [ ]:
bat_optimizer = BatAlgorithmContractOptimizer(
    n_tiers=5,
    n_bats=40,
    max_iterations=500,
)

print("=== BAT-INSPIRED SEARCH FOR CONTRACT OPTIMIZATION ===")
print(f"Supply-chain tiers: {bat_optimizer.n_tiers}")
print(f"Bat population: {bat_optimizer.n_bats}")
print(f"Contract parameters: {bat_optimizer.n_parameters}")
print(
    f"Frequency range: [{bat_optimizer.frequency_min}, "
    f"{bat_optimizer.frequency_max}]"
)
print(f"Initial loudness: {bat_optimizer.initial_loudness}")
print(f"Initial pulse rate: {bat_optimizer.initial_pulse_rate}\n")

optimal_params, optimal_fitness = bat_optimizer.optimize()
assert bat_optimizer._check_parameter_bounds(optimal_params)
assert np.isclose(
    optimal_fitness,
    bat_optimizer._evaluate_fitness(optimal_params),
    rtol=1e-10,
    atol=1e-12,
)

## Analyze and visualize the main run

The plots summarize the best fitness, the population's raw pairwise diversity, parameter-group values, and modeled cost components. A convergence curve that becomes flat indicates that the best recorded fitness has changed little; it does not prove global optimality. The information-sharing chart reports group means, while the individual decision vector remains available in `results`.

In [ ]:
results = bat_optimizer.analyze_results(optimal_params)

fig, axes = plt.subplots(3, 3, figsize=(20, 16))
fig.suptitle(
    "Bat-Inspired Multi-Party Contract Optimization",
    fontsize=16,
    fontweight="bold",
)

# Best fitness is positive under this model, so a logarithmic vertical scale is valid.
ax1 = axes[0, 0]
iterations = range(1, len(bat_optimizer.convergence_history) + 1)
ax1.plot(
    iterations,
    bat_optimizer.convergence_history,
    color="blue",
    linewidth=2,
    alpha=0.7,
)
ax1.set_xlabel("Iteration")
ax1.set_ylabel("Best fitness (model units)")
ax1.set_title("Best-Fitness History")
ax1.set_yscale("log")
ax1.grid(True, alpha=0.3)

ax2 = axes[0, 1]
ax2.plot(
    iterations,
    bat_optimizer.diversity_history,
    color="green",
    linewidth=2,
    alpha=0.7,
)
ax2.set_xlabel("Iteration")
ax2.set_ylabel("Mean pairwise Euclidean distance")
ax2.set_title("Raw Population Diversity")
ax2.grid(True, alpha=0.3)

ax3 = axes[0, 2]
sharing_types = ["Downstream", "Upstream"]
sharing_values = [results["downstream_sharing"], results["upstream_sharing"]]
sharing_colors = ["#FF6B6B", "#4ECDC4"]
bars = ax3.bar(sharing_types, sharing_values, color=sharing_colors, alpha=0.8)
ax3.set_ylabel("Mean information-sharing coefficient")
ax3.set_title(
    "Information Sharing\n"
    f"Downstream/upstream ratio: "
    f"{results['downstream_sharing'] / results['upstream_sharing']:.2f}"
)
ax3.set_ylim(0, 1)
ax3.grid(True, alpha=0.3, axis="y")
for bar, value in zip(bars, sharing_values):
    ax3.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.02,
        f"{value:.3f}",
        ha="center",
        va="bottom",
        fontweight="bold",
    )

ax4 = axes[1, 0]
tier_names = ["T1-2", "T2-3", "T3-4", "T4-5", "Cross-tier"]
ax4.bar(tier_names, results["cost_allocation"], color="purple", alpha=0.7)
ax4.set_ylabel("Cost-allocation parameter")
ax4.set_title("Cost-Allocation Parameters")
ax4.grid(True, alpha=0.3, axis="y")
ax4.tick_params(axis="x", rotation=35)

ax5 = axes[1, 1]
scenarios = ["Baseline", "Modeled contract"]
amplifications = [4.1, results["amplification"]]
bars = ax5.bar(
    scenarios,
    amplifications,
    color=["red", "green"],
    alpha=0.7,
)
ax5.set_ylabel("Bullwhip amplification factor")
ax5.set_title(
    f"Modeled Amplification\n"
    f"Reduction from baseline: {results['reduction_percentage']:.1f}%"
)
ax5.grid(True, alpha=0.3, axis="y")
for bar, amplification in zip(bars, amplifications):
    ax5.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.1,
        f"{amplification:.2f}x",
        ha="center",
        va="bottom",
        fontweight="bold",
    )

ax6 = axes[1, 2]
param_categories = [
    "Information\nsharing",
    "Cost\nallocation",
    "Penalties",
    "Capacity",
    "Timing",
]
param_means = [
    np.mean(results["info_sharing"]),
    np.mean(results["cost_allocation"]),
    np.mean(results["penalty_structures"]),
    np.mean(results["capacity_terms"]),
    np.mean(results["timing_params"]),
]
ax6.bar(
    param_categories,
    param_means,
    color=["#FF6B6B", "#4ECDC4", "#45B7D1", "#96CEB4", "#FFEAA7"],
    alpha=0.8,
)
ax6.set_ylabel("Mean parameter value")
ax6.set_title("Parameter-Group Means")
ax6.grid(True, alpha=0.3, axis="y")
ax6.tick_params(axis="x", rotation=25)

ax7 = axes[2, 0]
ax7.hist(
    bat_optimizer.frequencies,
    bins=20,
    color="orange",
    alpha=0.7,
    edgecolor="black",
)
ax7.set_xlabel("Frequency")
ax7.set_ylabel("Number of bats")
ax7.set_title("Final Frequency Distribution")
ax7.grid(True, alpha=0.3)

ax8 = axes[2, 1]
bat_indices = range(len(bat_optimizer.loudness))
ax8.scatter(
    bat_indices,
    bat_optimizer.loudness,
    alpha=0.6,
    color="blue",
    label="Loudness",
    s=30,
)
ax8.scatter(
    bat_indices,
    bat_optimizer.pulse_rates,
    alpha=0.6,
    color="red",
    label="Pulse rate",
    s=30,
)
ax8.set_xlabel("Bat index")
ax8.set_ylabel("Algorithm parameter value")
ax8.set_title("Final Loudness and Pulse Rates")
ax8.legend()
ax8.grid(True, alpha=0.3)

ax9 = axes[2, 2]
cost_categories = [
    "Information\nsystems",
    "Cost\nallocation",
    "Penalty\nsystems",
    "Capacity\nreservation",
    "Timing\nsystems",
]
cost_components = [
    np.sum(results["info_sharing"] ** 1.5) * 80_000,
    np.sum(results["cost_allocation"]) * 120_000,
    np.sum(results["penalty_structures"] ** 1.2) * 100_000,
    np.sum(results["capacity_terms"]) * 150_000,
    np.sum(results["timing_params"]) * 80_000,
]
assert np.isclose(
    sum(cost_components),
    results["impl_cost"],
    rtol=1e-10,
    atol=1e-8,
)
ax9.pie(
    cost_components,
    labels=cost_categories,
    colors=["#FF6B6B", "#4ECDC4", "#45B7D1", "#96CEB4", "#FFEAA7"],
    autopct="%1.1f%%",
    startangle=90,
)
ax9.set_title(f"Modeled Cost Breakdown\nTotal: ${results['impl_cost']:,.0f}")

plt.tight_layout()
plt.show()
print("Visualization complete.")

## Population-size sensitivity analysis

The following experiment runs the same search procedure with several population sizes, each using a 300-iteration limit. Because the global random-number generator advances between runs, these are sequential stochastic experiments rather than repeated trials with controlled, identical random streams. Differences can reflect both population size and randomness; interpret them as observations from these runs, not as a general ranking of population sizes.

The final bar chart uses within-metric min–max scores to place fitness, amplification, cost, and iteration count on a common 0–100 display. For each metric, a higher score denotes the better value among the tested population sizes. If a metric is identical across all sizes, its scores are set to 50 because that metric does not distinguish the tested cases. The scores are descriptive and are not part of the optimization objective.

In [ ]:
print("=== POPULATION-SIZE SENSITIVITY ANALYSIS ===")
print("Comparing the specified bat population sizes.\n")

population_sizes = [20, 30, 40, 50, 60]
population_results = []

for pop_size in population_sizes:
    print(f"Testing population size: {pop_size} bats...")
    test_optimizer = BatAlgorithmContractOptimizer(
        n_tiers=5,
        n_bats=pop_size,
        max_iterations=300,
    )
    test_params, test_fitness = test_optimizer.optimize()

    test_amplification = test_optimizer._calculate_bullwhip_amplification(
        test_params
    )
    test_cost = test_optimizer._calculate_implementation_cost(test_params)
    test_iterations = len(test_optimizer.convergence_history)
    test_diversity = (
        test_optimizer.diversity_history[-1]
        if test_optimizer.diversity_history
        else 0.0
    )
    population_results.append(
        {
            "population_size": pop_size,
            "final_fitness": test_fitness,
            "amplification": test_amplification,
            "cost": test_cost,
            "iterations": test_iterations,
            "final_diversity": test_diversity,
        }
    )

    print(f"  Final fitness: {test_fitness:.6f}")
    print(f"  Amplification: {test_amplification:.3f}x")
    print(f"  Modeled cost: ${test_cost:,.0f}")
    print(f"  Recorded iterations: {test_iterations}\n")

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle("Population-Size Sensitivity Analysis", fontsize=16, fontweight="bold")

fitness_values = [result["final_fitness"] for result in population_results]
amplification_values = [
    result["amplification"] for result in population_results
]
cost_values = [result["cost"] for result in population_results]
iteration_values = [result["iterations"] for result in population_results]
diversity_values = [
    result["final_diversity"] for result in population_results
]

ax1 = axes[0, 0]
ax1.plot(population_sizes, fitness_values, "bo-", linewidth=2, markersize=8)
ax1.set_xlabel("Population size (bats)")
ax1.set_ylabel("Final fitness (model units)")
ax1.set_title("Fitness by Population Size")
ax1.grid(True, alpha=0.3)

ax2 = axes[0, 1]
ax2.plot(
    population_sizes,
    amplification_values,
    "ro-",
    linewidth=2,
    markersize=8,
)
ax2.set_xlabel("Population size (bats)")
ax2.set_ylabel("Bullwhip amplification factor")
ax2.set_title("Amplification by Population Size")
ax2.grid(True, alpha=0.3)

ax3 = axes[0, 2]
ax3.plot(population_sizes, cost_values, "go-", linewidth=2, markersize=8)
ax3.set_xlabel("Population size (bats)")
ax3.set_ylabel("Modeled implementation cost ($)")
ax3.set_title("Cost by Population Size")
ax3.grid(True, alpha=0.3)

ax4 = axes[1, 0]
ax4.plot(
    population_sizes,
    iteration_values,
    "mo-",
    linewidth=2,
    markersize=8,
)
ax4.set_xlabel("Population size (bats)")
ax4.set_ylabel("Iterations recorded")
ax4.set_title("Recorded Iterations by Population Size")
ax4.grid(True, alpha=0.3)

ax5 = axes[1, 1]
ax5.plot(
    population_sizes,
    diversity_values,
    "co-",
    linewidth=2,
    markersize=8,
)
ax5.set_xlabel("Population size (bats)")
ax5.set_ylabel("Final mean pairwise distance")
ax5.set_title("Final Raw Diversity by Population Size")
ax5.grid(True, alpha=0.3)

def within_metric_score(values, higher_is_better=False):
    """Scale one metric to 0–100 without dividing by a zero range."""
    values = np.asarray(values, dtype=float)
    value_range = np.max(values) - np.min(values)
    if np.isclose(value_range, 0.0):
        return np.full(values.shape, 50.0)
    scores = (values - np.min(values)) / value_range * 100.0
    return scores if higher_is_better else 100.0 - scores

fitness_scores = within_metric_score(fitness_values)
amplification_scores = within_metric_score(amplification_values)
cost_scores = within_metric_score(cost_values)
speed_scores = within_metric_score(iteration_values)
average_scores = np.mean(
    [fitness_scores, amplification_scores, cost_scores, speed_scores],
    axis=0,
)

ax6 = axes[1, 2]
ax6.bar(
    [str(size) for size in population_sizes],
    average_scores,
    color="#45B7D1",
    alpha=0.8,
)
ax6.set_xlabel("Population size (bats)")
ax6.set_ylabel("Mean within-metric score (%)")
ax6.set_title("Descriptive Normalized Summary")
ax6.set_ylim(0, 100)
ax6.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

best_result = min(population_results, key=lambda result: result["final_fitness"])
print("\nBEST OBSERVED FITNESS IN THIS EXPERIMENT:")
print(f"Population size: {best_result['population_size']} bats")
print(f"Fitness: {best_result['final_fitness']:.6f}")
print(f"Amplification: {best_result['amplification']:.3f}x")
print(f"Modeled cost: ${best_result['cost']:,.0f}")
print("This identifies the best observed run among the tested sizes, not a universal optimum.")

## Position of this tier in the learning sequence

This tier presents a population-based stochastic search for a higher-dimensional contract model. Its population can hold candidate solutions in different parts of the bounded parameter space, while frequency-dependent movement and local refinement provide the search mechanics used in this notebook. The code also records diversity so that readers can inspect whether the population remains spread out or becomes concentrated.

These design features do not guarantee escape from every local optimum or establish that the returned candidate is globally optimal. Performance depends on the objective, parameter bounds, algorithm settings, and random sequence. The notebook does not implement a Tier 2 method, so it does not make an empirical performance comparison with that method.

### When this approach is relevant

A population-based heuristic can be considered when a contract model has many interacting continuous parameters and a good solution is difficult to locate with direct inspection. It is also useful as a teaching example of how exploration, exploitation, local refinement, and convergence diagnostics fit together. The model still needs validation against the decision-maker's real contractual, financial, and operational requirements.

## Reading the results and recognizing limitations

### What the experiment can show

- The convergence history shows how the best fitness found so far changes during a run.
- The diversity history describes the spread of the candidate vectors in their raw parameter coordinates.
- The parameter summaries and plots expose the candidate selected by the search and its modeled amplification and cost.
- The population-size experiment compares observed runs under the stated settings. Stochastic variation means that a single run for each size is not a statistical performance study.

### What the experiment cannot establish

- The model does not establish that the chosen coefficients cause the reported operational effects in a real supply chain.
- The cost term is inactive within the specified parameter bounds, so the current fitness does not reward cost reductions below the threshold. Cost is reported, but the search is not making an active cost-versus-amplification trade-off.
- The model's cost, demand, holding-cost, stockout-cost, and procurement-cost descriptors are not all used in the fitness calculation. In particular, demand and the listed tier cost arrays are retained as context but do not enter the amplification or implementation-cost formulas.
- The information-sharing values produced by a run need not exhibit any particular downstream/upstream pattern. The asymmetry in the model is in the weighting of the two groups, not a constraint requiring one group to have larger values.
- A low-diversity population or a flat best-fitness history is a diagnostic observation, not a proof of convergence to a global optimum.

Further work within this modeling approach could validate the response functions with supply-chain data, add decision-relevant feasibility constraints, and repeat stochastic experiments across multiple seeds. Those extensions would require explicit modeling choices and are not implemented in this notebook.